In [2]:
# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.decomposition import PCA

# Step 2: Load dataset
df = pd.read_csv("heart.csv")

print("First 5 rows:")
print(df.head())

# Step 3: Identify categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
print("Categorical Columns:", categorical_cols)

# Step 4: Apply Label Encoding (for binary categories)
label_encoders = {}

for col in categorical_cols:
    if df[col].nunique() == 2:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])
        label_encoders[col] = le

# Step 5: One-Hot Encoding (for multi-category columns)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Step 6: Split features and target
X = df.drop("HeartDisease", axis=1)   # change if your target column name is different
y = df["HeartDisease"]

# Step 7: Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Step 8: Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 9: Train models (WITHOUT PCA)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier()
}

print("\nAccuracy WITHOUT PCA:")
results_without_pca = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results_without_pca[name] = acc
    print(f"{name}: {acc:.4f}")

# Step 10: Apply PCA
pca = PCA(n_components=0.95)  # keep 95% variance

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("\nNumber of components after PCA:", pca.n_components_)

# Step 11: Train models (WITH PCA)

print("\nAccuracy WITH PCA:")
results_with_pca = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    results_with_pca[name] = acc
    print(f"{name}: {acc:.4f}")

# Step 12: Compare Results

print("\nFinal Comparison:")
for model in models.keys():
    print(f"{model} -> Without PCA: {results_without_pca[model]:.4f}, With PCA: {results_with_pca[model]:.4f}")

First 5 rows:
   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  
Categorical Columns: ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina', 'ST_Slope']

Accuracy WITHOUT PCA:
Logistic Regression: 0.8533
SVM: 0.8750
Random Forest: 0.8750

Number of compone